In [1]:
# nyiso_vs_ours.py — score our Day-1 forecast against NYISO's own, on the same days.
# Runs anywhere with internet + pandas. No GPU, no CAIR.
#   pip install pandas requests
import io, zipfile, requests
import numpy as np, pandas as pd

OURS = ("https://raw.githubusercontent.com/Sangi2805/Forecasting-Energy-Demand"
        "/main/reports/tft_zonal_predictions.csv")          # or a local path
ZONES = ["CAPITL","CENTRL","DUNWOD","GENESE","HUD VL","LONGIL",
         "MHK VL","MILLWD","N.Y.C.","NORTH","WEST"]

def norm(c): return " ".join(str(c).strip().upper().split())

def fetch_month(year, month):
    """NYISO publishes isolf as monthly zips of daily files."""
    url = f"http://mis.nyiso.com/public/csv/isolf/{year}{month:02d}01isolf_csv.zip"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    rows = []
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        for name in z.namelist():
            if not name.lower().endswith(".csv"):
                continue
            issue = pd.to_datetime(name[:8], format="%Y%m%d").date()
            df = pd.read_csv(z.open(name))
            ts = pd.to_datetime(df["Time Stamp"])
            cols = {norm(c): c for c in df.columns if c != "Time Stamp"}
            have = [cols[z_] for z_ in ZONES if z_ in cols]
            if not have:
                continue
            total = df[have].apply(pd.to_numeric, errors="coerce").sum(axis=1)
            rows.append(pd.DataFrame({"issue": issue, "ts": ts, "mw": total}))
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

def main():
    ours = pd.read_csv(OURS, parse_dates=["date"])
    ours["d"] = ours["date"].dt.date
    months = sorted({(d.year, d.month) for d in ours["d"]})
    print(f"our rows: {len(ours)}, months to pull: {len(months)}")

    parts = []
    for y, m in months:
        try:
            p = fetch_month(y, m)
            if len(p): parts.append(p)
            print(f"  {y}-{m:02d}: {len(p):,} rows")
        except Exception as e:
            print(f"  {y}-{m:02d}: skipped ({e})")
    ny = pd.concat(parts, ignore_index=True)
    ny["target"] = ny["ts"].dt.date

    # how far ahead does each file forecast? tells us the file structure
    ny["lead"] = (pd.to_datetime(ny["target"]) - pd.to_datetime(ny["issue"])).dt.days
    print("\nlead-day distribution in NYISO files:")
    print(ny["lead"].value_counts().sort_index().to_string())

    # day-ahead = forecast issued the day before the target date
    da = ny[ny["lead"] == 1]
    daily = da.groupby("target")["mw"].sum().rename("nyiso_pred")

    j = ours.set_index("d").join(daily, how="inner")
    j = j.dropna(subset=["nyiso_pred", "pred_day1", "actual_day1"])
    if j.empty:
        raise SystemExit("no overlapping days — check the lead distribution above")

    a = j["actual_day1"]
    ours_mape  = float((np.abs(a - j["pred_day1"])  / a).mean() * 100)
    nyiso_mape = float((np.abs(a - j["nyiso_pred"]) / a).mean() * 100)

    print(f"\n=== Day-ahead, daily totals, {len(j)} matched days "
          f"({j.index.min()} to {j.index.max()}) ===")
    print(f"  our model : {ours_mape:.2f}%")
    print(f"  NYISO     : {nyiso_mape:.2f}%")
    j.to_csv("nyiso_comparison.csv")
    print("per-day detail -> nyiso_comparison.csv")

if __name__ == "__main__":
    main()

our rows: 854, months to pull: 29
  2024-01: 4,464 rows
  2024-02: 4,176 rows
  2024-03: 4,458 rows
  2024-04: 4,320 rows
  2024-05: 4,464 rows
  2024-06: 4,320 rows
  2024-07: 4,464 rows
  2024-08: 4,464 rows
  2024-09: 4,320 rows
  2024-10: 4,467 rows
  2024-11: 4,323 rows
  2024-12: 4,464 rows
  2025-01: 4,464 rows
  2025-02: 4,008 rows
  2025-03: 4,458 rows
  2025-04: 4,320 rows
  2025-05: 4,440 rows
  2025-06: 4,320 rows
  2025-07: 4,464 rows
  2025-08: 4,464 rows
  2025-09: 4,320 rows
  2025-10: 4,468 rows
  2025-11: 4,322 rows
  2025-12: 4,464 rows
  2026-01: 4,464 rows
  2026-02: 4,032 rows
  2026-03: 4,434 rows
  2026-04: 4,320 rows
  2026-05: 4,464 rows

lead-day distribution in NYISO files:
lead
0    21167
1    21167
2    21167
3    21167
4    21167
5    21095

=== Day-ahead, daily totals, 854 matched days (2024-01-24 to 2026-05-26) ===
  our model : 1.43%
  NYISO     : 2.69%
per-day detail -> nyiso_comparison.csv


In [2]:
# check_and_all_leads.py — verify the NYISO parse, then compare every horizon.
import io, zipfile, requests
import numpy as np, pandas as pd

OURS = ("https://raw.githubusercontent.com/Sangi2805/Forecasting-Energy-Demand"
        "/main/reports/tft_zonal_predictions.csv")
ZONES = ["CAPITL","CENTRL","DUNWOD","GENESE","HUD VL","LONGIL",
         "MHK VL","MILLWD","N.Y.C.","NORTH","WEST"]
def norm(c): return " ".join(str(c).strip().upper().split())

rows, missing = [], {}
ours = pd.read_csv(OURS, parse_dates=["date"]); ours["d"] = ours["date"].dt.date
for y, m in sorted({(d.year, d.month) for d in ours["d"]}):
    try:
        r = requests.get(f"http://mis.nyiso.com/public/csv/isolf/{y}{m:02d}01isolf_csv.zip",
                         timeout=60); r.raise_for_status()
    except Exception as e:
        print(f"{y}-{m:02d} skipped ({e})"); continue
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        for name in [n for n in z.namelist() if n.lower().endswith(".csv")]:
            issue = pd.to_datetime(name[:8], format="%Y%m%d").date()
            df = pd.read_csv(z.open(name))
            cols = {norm(c): c for c in df.columns if c != "Time Stamp"}
            have = [c for c in ZONES if c in cols]
            if len(have) < 11:                      # <-- the bug check
                missing[name] = sorted(set(ZONES) - set(have))
            if not have: continue
            tot = df[[cols[c] for c in have]].apply(pd.to_numeric, errors="coerce").sum(axis=1)
            rows.append(pd.DataFrame({"issue": issue,
                                      "ts": pd.to_datetime(df["Time Stamp"]),
                                      "mw": tot, "nz": len(have)}))

ny = pd.concat(rows, ignore_index=True)
print(f"\nfiles with fewer than 11 zones: {len(missing)}")
for k, v in list(missing.items())[:5]: print("   ", k, "missing", v)

ny["target"] = ny["ts"].dt.date
ny["lead"] = (pd.to_datetime(ny["target"]) - pd.to_datetime(ny["issue"])).dt.days

print(f"\n{'':6} {'ours':>8} {'NYISO':>8} {'days':>7}   (daily totals, full-zone files only)")
for d in range(1, 6):
    daily = (ny[(ny["lead"] == d) & (ny["nz"] == 11)]
             .groupby("target")["mw"].sum().rename("ny"))
    j = ours.set_index("d").join(daily, how="inner").dropna(
        subset=["ny", f"pred_day{d}", f"actual_day{d}"])
    if j.empty: print(f"day{d}   no overlap"); continue
    a = j[f"actual_day{d}"]
    o = float((np.abs(a - j[f"pred_day{d}"]) / a).mean() * 100)
    n = float((np.abs(a - j["ny"]) / a).mean() * 100)
    med = float((np.abs(a - j["ny"]) / a).median() * 100)
    print(f"day{d}  {o:8.2f} {n:8.2f} {len(j):7d}   NYISO median {med:.2f}%")


files with fewer than 11 zones: 0

           ours    NYISO    days   (daily totals, full-zone files only)
day1      1.43     2.69     854   NYISO median 2.43%
day2      1.84     5.06     854   NYISO median 4.01%
day3      2.11     7.17     854   NYISO median 6.04%
day4      2.58     8.08     854   NYISO median 6.68%
day5      2.95     8.52     851   NYISO median 7.31%


In [3]:
# nyiso_fixed.py — correct target alignment + bias check.
import io, zipfile, requests
import numpy as np, pandas as pd

OURS = ("https://raw.githubusercontent.com/Sangi2805/Forecasting-Energy-Demand"
        "/main/reports/tft_zonal_predictions.csv")
ZONES = ["CAPITL","CENTRL","DUNWOD","GENESE","HUD VL","LONGIL",
         "MHK VL","MILLWD","N.Y.C.","NORTH","WEST"]
def norm(c): return " ".join(str(c).strip().upper().split())

ours = pd.read_csv(OURS, parse_dates=["date"])

# our predictions in long form: pred_dayN on row D targets D + (N-1)
long = []
for d in range(1, 6):
    t = ours[["date", f"pred_day{d}", f"actual_day{d}"]].copy()
    t.columns = ["issue_date", "our_pred", "actual"]
    t["lead"] = d
    t["target"] = (t["issue_date"] + pd.Timedelta(days=d - 1)).dt.date
    long.append(t)
long = pd.concat(long, ignore_index=True)

rows = []
for y, m in sorted({(d.year, d.month) for d in ours["date"].dt.date}):
    try:
        r = requests.get(f"http://mis.nyiso.com/public/csv/isolf/{y}{m:02d}01isolf_csv.zip",
                         timeout=60); r.raise_for_status()
    except Exception as e:
        print(f"{y}-{m:02d} skipped ({e})"); continue
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        for name in [n for n in z.namelist() if n.lower().endswith(".csv")]:
            issue = pd.to_datetime(name[:8], format="%Y%m%d").date()
            df = pd.read_csv(z.open(name))
            cols = {norm(c): c for c in df.columns if c != "Time Stamp"}
            have = [cols[c] for c in ZONES if c in cols]
            if len(have) < 11: continue
            tot = df[have].apply(pd.to_numeric, errors="coerce").sum(axis=1)
            rows.append(pd.DataFrame({"issue": issue,
                                      "ts": pd.to_datetime(df["Time Stamp"]), "mw": tot}))

ny = pd.concat(rows, ignore_index=True)
ny["target"] = ny["ts"].dt.date
ny["lead"] = (pd.to_datetime(ny["target"]) - pd.to_datetime(ny["issue"])).dt.days
nyd = ny.groupby(["target", "lead"])["mw"].sum().rename("ny_pred").reset_index()

j = long.merge(nyd, on=["target", "lead"], how="inner").dropna()
print(f"{'':6} {'ours':>7} {'NYISO':>7} {'days':>6}  {'our bias':>9} {'NY bias':>9}")
for d in range(1, 6):
    g = j[j["lead"] == d]
    if g.empty: print(f"day{d}  no overlap"); continue
    a = g["actual"]
    o  = float((np.abs(a - g["our_pred"]) / a).mean() * 100)
    n  = float((np.abs(a - g["ny_pred"]) / a).mean() * 100)
    ob = float(((g["our_pred"] - a) / a).mean() * 100)
    nb = float(((g["ny_pred"] - a) / a).mean() * 100)
    print(f"day{d} {o:7.2f} {n:7.2f} {len(g):6d}  {ob:8.2f}% {nb:8.2f}%")
j.to_csv("nyiso_fixed.csv", index=False)

          ours   NYISO   days   our bias   NY bias
day1    1.43    2.69    854     -0.07%    -2.35%
day2    1.84    2.90    854     -0.72%    -2.31%
day3    2.11    3.16    854     -0.79%    -2.29%
day4    2.58    3.50    854     -0.79%    -2.28%
day5    2.96    3.87    851     -0.79%    -2.26%


In [4]:
# actual_check.py — is our "actual" the same series NYISO forecasts against?
import io, zipfile, requests
import numpy as np, pandas as pd

OURS = ("https://raw.githubusercontent.com/Sangi2805/Forecasting-Energy-Demand"
        "/main/reports/tft_zonal_predictions.csv")
ours = pd.read_csv(OURS, parse_dates=["date"])
ours["d"] = ours["date"].dt.date

rows = []
for y, m in sorted({(d.year, d.month) for d in ours["d"]}):
    try:
        r = requests.get(f"http://mis.nyiso.com/public/csv/palIntegrated/"
                         f"{y}{m:02d}01palIntegrated_csv.zip", timeout=60)
        r.raise_for_status()
    except Exception as e:
        print(f"{y}-{m:02d} skipped ({e})"); continue
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        for name in [n for n in z.namelist() if n.lower().endswith(".csv")]:
            df = pd.read_csv(z.open(name))
            lc = [c for c in df.columns if "load" in c.lower()][0]
            df["ts"] = pd.to_datetime(df["Time Stamp"])
            rows.append(df[["ts", lc]].rename(columns={lc: "mw"}))
    print(f"  {y}-{m:02d} ok")

act = pd.concat(rows, ignore_index=True)
act["mw"] = pd.to_numeric(act["mw"], errors="coerce")
daily = act.groupby(act["ts"].dt.date)["mw"].sum().rename("nyiso_actual")

j = ours.set_index("d").join(daily, how="inner").dropna(
    subset=["actual_day1", "nyiso_actual"])
gap = (j["actual_day1"] - j["nyiso_actual"]) / j["nyiso_actual"] * 100
print(f"\nmatched days: {len(j)}")
print(f"our actual vs NYISO actual: mean {gap.mean():+.2f}%  "
      f"median {gap.median():+.2f}%  sd {gap.std():.2f}%")
print("\nIf mean is near 0, the two series agree and NYISO's forecast really runs low.")
print("If mean is near +2.3%, our ground truth is the outlier and all scoring must be redone.")

  2024-01 ok
  2024-02 ok
  2024-03 ok
  2024-04 ok
  2024-05 ok
  2024-06 ok
  2024-07 ok
  2024-08 ok
  2024-09 ok
  2024-10 ok
  2024-11 ok
  2024-12 ok
  2025-01 ok
  2025-02 ok
  2025-03 ok
  2025-04 ok
  2025-05 ok
  2025-06 ok
  2025-07 ok
  2025-08 ok
  2025-09 ok
  2025-10 ok
  2025-11 ok
  2025-12 ok
  2026-01 ok
  2026-02 ok
  2026-03 ok
  2026-04 ok
  2026-05 ok

matched days: 854
our actual vs NYISO actual: mean +0.01%  median +0.00%  sd 0.31%

If mean is near 0, the two series agree and NYISO's forecast really runs low.
If mean is near +2.3%, our ground truth is the outlier and all scoring must be redone.


In [5]:
# debias.py — compare raw and bias-corrected. Run after nyiso_fixed.py.
import numpy as np, pandas as pd

j = pd.read_csv("nyiso_fixed.csv")
print(f"{'':5} {'--- raw ---':>18}   {'--- bias-corrected ---':>24}")
print(f"{'':5} {'ours':>8} {'NYISO':>8}   {'ours':>8} {'NYISO':>8}  {'days':>6}")
for d in range(1, 6):
    g = j[j["lead"] == d]
    if g.empty: continue
    a = g["actual"].to_numpy()
    for col, lab in (("our_pred", "ours"), ("ny_pred", "nyiso")):
        pass
    eo = (g["our_pred"].to_numpy() - a) / a
    en = (g["ny_pred"].to_numpy()  - a) / a
    raw_o, raw_n = np.abs(eo).mean()*100, np.abs(en).mean()*100
    db_o = np.abs(eo - eo.mean()).mean()*100
    db_n = np.abs(en - en.mean()).mean()*100
    print(f"day{d} {raw_o:8.2f} {raw_n:8.2f}   {db_o:8.2f} {db_n:8.2f}  {len(g):6d}")

             --- raw ---     --- bias-corrected ---
          ours    NYISO       ours    NYISO    days
day1     1.43     2.69       1.43     1.77     854
day2     1.84     2.90       1.77     2.07     854
day3     2.11     3.16       2.01     2.40     854
day4     2.58     3.50       2.48     2.85     854
day5     2.96     3.87       2.87     3.26     851
